In [ ]:
# Setup: change working directory to project root so all relative paths work
import os
from pathlib import Path

project_root = str(Path(os.getcwd()).parent) if Path(os.getcwd()).name == "notebooks" else os.getcwd()
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

In [47]:
import json
import pandas as pd

# Load the JSON file
with open('data/raw/ddse_exp_ea.json', 'r') as f:
    data = json.load(f)

# Access the different components
customdata = data['customdata']
x_values = data['x']  
y_values = data['y']

print(f"Number of traces: {len(customdata)}")
print(f"Structure: customdata={len(customdata)}, x={len(x_values)}, y={len(y_values)}")

# Create a DataFrame from the customdata, x_values, and y_values
df_ea = pd.DataFrame({
    'customdata': customdata,
    'Temp_K': x_values,
    'Ea_eV': y_values
})

df_ea.head()

Number of traces: 3220
Structure: customdata=3220, x=3220, y=3220


,customdata,Temp_K,Ea_eV
0,"[Li2(BH4)(NH2), Li, 10.1021/ja907249p, 0.014, ...",311.167813,0.353580
1,"[Li2(BH4)(NH2), Li, 10.1021/ja907249p, 0.014, ...",305.581445,0.353580
2,"[Li4(BH4)(NH2)3, Li, 10.1021/ja907249p, 0.0233...",308.511524,0.360144
3,"[Li4(BH4)(NH2)3, Li, 10.1021/ja907249p, 0.0233...",303.278440,0.360144
4,"[Li4(BH4)(NH2)3, Li, 10.1021/ja907249p, 0.0233...",301.175790,0.360144


In [48]:
def customdata_handler(df):
    """
    Custom handler to process DataFrame columns.
    """
    if 'customdata' in df.columns:
        # convert 'customdata' column from string to list
        df['customdata'] = df['customdata'].apply(lambda x: eval(x) if isinstance(x, str) else x)
        # print("Processing 'customdata' column...")
        # print(f"customdata = {df['customdata'].head()}")
        df['electrolyte'] = df['customdata'].apply(lambda x: x[0] if len(x) > 0 else None)
        df['electrolyte'] = df['electrolyte'].apply(lambda x: x if isinstance(x, str) else None)
        # print(f"electrolyte = {df['electrolyte'].head()}")

        df['doi'] = df['customdata'].apply(lambda x: x[2] if len(x) > 2 else None)
        df['doi'] = df['doi'].apply(lambda x: x if isinstance(x, str) else None)
        # print(f"doi = {df['doi'].head()}")

        df.drop(columns=['customdata'], inplace=True, errors='ignore')
    else:
        print("No 'customdata' column found in the DataFrame.")
    return df

In [49]:
df_ea = customdata_handler(df_ea)
df_ea.head()

,Temp_K,Ea_eV,electrolyte,doi
0,311.167813,0.353580,Li2(BH4)(NH2),10.1021/ja907249p
1,305.581445,0.353580,Li2(BH4)(NH2),10.1021/ja907249p
2,308.511524,0.360144,Li4(BH4)(NH2)3,10.1021/ja907249p
3,303.278440,0.360144,Li4(BH4)(NH2)3,10.1021/ja907249p
4,301.175790,0.360144,Li4(BH4)(NH2)3,10.1021/ja907249p


In [50]:
import json
import pandas as pd

def process_plotly_json(json_file, y_label):
    """
    Process plotly JSON data and return a cleaned DataFrame.
    
    Parameters:
    json_file (str): Path to the JSON file
    y_label (str): Label for the y-axis data column
    
    Returns:
    pd.DataFrame: Processed DataFrame with electrolyte, doi, Temp_K, and y-data columns
    """
    
    # Load the JSON file
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Access the different components
    customdata = data['customdata']
    x_values = data['x']  
    y_values = data['y']
    
    print(f"Number of traces: {len(customdata)}")
    print(f"Structure: customdata={len(customdata)}, x={len(x_values)}, y={len(y_values)}")
    
    # Create a DataFrame from the customdata, x_values, and y_values
    df = pd.DataFrame({
        'customdata': customdata,
        'Temp_K': x_values,
        y_label: y_values
    })
    
    # Process customdata column
    if 'customdata' in df.columns:
        # Convert 'customdata' column from string to list if needed
        df['customdata'] = df['customdata'].apply(lambda x: eval(x) if isinstance(x, str) else x)
        
        # Extract electrolyte (first element)
        df['electrolyte'] = df['customdata'].apply(lambda x: x[0] if len(x) > 0 else None)
        df['electrolyte'] = df['electrolyte'].apply(lambda x: x if isinstance(x, str) else None)
        
        # Extract DOI (third element)
        df['doi'] = df['customdata'].apply(lambda x: x[2] if len(x) > 2 else None)
        df['doi'] = df['doi'].apply(lambda x: x if isinstance(x, str) else None)
        
        # Drop the customdata column
        df.drop(columns=['customdata'], inplace=True, errors='ignore')
    else:
        print("No 'customdata' column found in the DataFrame.")
    
    return df

# Usage examples:
df_ea = process_plotly_json('data/raw/ddse_exp_ea.json', 'Ea_eV')
df_ic = process_plotly_json('data/raw/ddse_exp_ic.json', 'Ionic_Conductivity')
df_type = process_plotly_json('data/raw/ddse_exp_type.json', 'Material_Type')


Number of traces: 3220
Structure: customdata=3220, x=3220, y=3220
Number of traces: 3220
Structure: customdata=3220, x=3220, y=3220
Number of traces: 3220
Structure: customdata=3220, x=3220, y=3220


In [52]:
# join the DataFrames on 'electrolyte' and 'doi'
df_combined = df_ea.merge(df_ic, on=['electrolyte', 'doi', 'Temp_K'], how='outer', suffixes=('_ea', '_ic'))
df_combined = df_combined.merge(df_type, on=['electrolyte', 'doi', 'Temp_K'], how='outer', suffixes=('', '_type'))
df_combined.head()

,Temp_K,Ea_eV,electrolyte,doi,Ionic_Conductivity,Material_Type
0,298.284118,0.242549,Li5.55(P0.95Ge0.05)S4.5Cl1.5,10.1016/j.jallcom.2025.180413,0.009822,Argyrodite
1,298.403194,0.236612,Li5.6(P0.9Ge0.1)S4.5Cl1.5,10.1016/j.jallcom.2025.180413,0.012864,Argyrodite
2,285.641893,0.437157,LiBiSbS2,10.2139/ssrn.5101260,0.000001,Sulfide
3,294.163872,0.437157,LiBiSbS2,10.2139/ssrn.5101260,0.000001,Sulfide
4,298.549601,0.437157,LiBiSbS2,10.2139/ssrn.5101260,0.000002,Sulfide


In [53]:
# save the combined DataFrame to a CSV file
df_combined.to_csv('data/processed/ddse_combined.csv', index=False)